# Lab 5


Matrix Representation: In this lab you will be creating a simple linear algebra system. In memory, we will represent matrices as nested python lists as we have done in lecture. In the exercises below, you are required to explicitly test every feature you implement, demonstrating it works.

1. Create a `matrix` class with the following properties:
    * It can be initialized in 2 ways:
        1. with arguments `n` and `m`, the size of the matrix. A newly instanciated matrix will contain all zeros.
        2. with a list of lists of values. Note that since we are using lists of lists to implement matrices, it is possible that not all rows have the same number of columns. Test explicitly that the matrix is properly specified.
    * Matrix instances `M` can be indexed with `M[i][j]` and `M[i,j]`.
    * Matrix assignment works in 2 ways:
        1. If `M_1` and `M_2` are `matrix` instances `M_1=M_2` sets the values of `M_1` to those of `M_2`, if they are the same size. Error otherwise.
        2. In example above `M_2` can be a list of lists of correct size.


2. Add the following methods:
    * `shape()`: returns a tuple `(n,m)` of the shape of the matrix.
    * `transpose()`: returns a new matrix instance which is the transpose of the matrix.
    * `row(n)` and `column(n)`: that return the nth row or column of the matrix M as a new appropriately shaped matrix object.
    * `to_list()`: which returns the matrix as a list of lists.
    *  `block(n_0,n_1,m_0,m_1)` that returns a smaller matrix located at the n_0 to n_1 columns and m_0 to m_1 rows. 
    * (Extra credit) Modify `__getitem__` implemented above to support slicing.
        

3. Write functions that create special matrices (note these are standalone functions, not member functions of your `matrix` class):
    * `constant(n,m,c)`: returns a `n` by `m` matrix filled with floats of value `c`.
    * `zeros(n,m)` and `ones(n,m)`: return `n` by `m` matrices filled with floats of value `0` and `1`, respectively.
    * `eye(n)`: returns the n by n identity matrix.

4. Add the following member functions to your class. Make sure to appropriately test the dimensions of the matrices to make sure the operations are correct.
    * `M.scalarmul(c)`: a matrix that is scalar product $cM$, where every element of $M$ is multiplied by $c$.
    * `M.add(N)`: adds two matrices $M$ and $N$. Don’t forget to test that the sizes of the matrices are compatible for this and all other operations.
    * `M.sub(N)`: subtracts two matrices $M$ and $N$.
    * `M.mat_mult(N)`: returns a matrix that is the matrix product of two matrices $M$ and $N$.
    * `M.element_mult(N)`: returns a matrix that is the element-wise product of two matrices $M$ and $N$.
    * `M.equals(N)`: returns true/false if $M==N$.

5. Overload python operators to appropriately use your functions in 4 and allow expressions like:
    * 2*M
    * M*2
    * M+N
    * M-N
    * M*N
    * M==N
    * M=N


6. Demonstrate the basic properties of matrices with your matrix class by creating two 2 by 2 example matrices using your Matrix class and illustrating the following:

$$
(AB)C=A(BC)
$$
$$
A(B+C)=AB+AC
$$
$$
AB\neq BA
$$
$$
AI=A
$$

In [17]:
from ast import Index
import random  

class Matrix:
    def __init__(self, n, m, seed=None):  #Have a seed option such that each time an instance is reloaded won't have to worry about values changing
        self.__n = n  
        self.__m = m  
        if seed is not None:
          random.seed(seed)

        self.__mat = [[random.randint(0, 9) for _ in range(self.__m)] for _ in range(self.__n)]  # Random values

    def shape(self):
        return (self.__n, self.__m)

    def __repr__(self):
      return "\n".join(" ".join(f"{val:3}" for val in row) for row in self.__mat)


    def transpose(self):
        transposed = [[self.__mat[j][i] for j in range(self.__n)] for i in range(self.__m)]
        return Matrix(self.__m, self.__n, seed=None)._set_matrix(transposed)

    def _set_matrix(self, new_matrix):
        """Helper method to replace the matrix after transpose."""
        self.__mat = new_matrix
        self.__n, self.__m = len(new_matrix), len(new_matrix[0])
        return self

    def get_row(self, n):
      n = n-1  ##Should re-index to 0-indexing base in Python
      if 0 <= n < self.__n:
        return self.__mat[n]
      else:
        raise IndexError("Row index out of range")

    def get_column(self, n):
      n = n-1
      if 0 <= n < self.__m:
        return [row[n] for row in self.__mat]
      else:
        raise IndexError("Column index out of range")


    def to_list(self):
      return self.__mat

    def block(self,n_0,n_1,m_0,m_1):
      n_0, n_1, m_0, m_1 = n_0 - 1, n_1, m_0-1, m_1

      if not (0 <= n_0 < self.__n and 0 < n_1 <= self.__n and
              0 <= m_0 < self.__m and 0 < m_1 <= self.__m):
        raise IndexError("Block indices are out of bounds")

      sub_matrix = [row[m_0:m_1] for row in self.__mat[n_0:n_1]]

      return Matrix(n_1 - n_0, m_1 - m_0)._set_matrix(sub_matrix)

    def scalarmul(self, c):
      new_matrix = [[self.__mat[i][j] * c for j in range(self.__m)] for i in range(self.__n)]
      return Matrix(self.__n, self.__m)._set_matrix(new_matrix)


    def add(self, N):
      if not isinstance(N, Matrix):
        raise TypeError("Expected input to be of type Matrix")

      if self.shape() != N.shape():
        raise ValueError("Matrices must have the same dimensions to be added")

      new_matrix = [[self.__mat[i][j] + N.__mat[i][j] for j in range(self.__m)] for i in range(self.__n)]

      return Matrix(self.__n, self.__m)._set_matrix(new_matrix)

    def sub(self, N):
      if not isinstance(N, Matrix):
        raise TypeError("Expected input to be of type Matrix")

      if self.shape() != N.shape():
        raise ValueError("Matrices must have the same dimensions to be subtracted")

      new_matrix = [[self.__mat[i][j] - N.__mat[i][j] for j in range(self.__m)] for i in range(self.__n)]

      return Matrix(self.__n, self.__m)._set_matrix(new_matrix)

    def element_mult(self,N):
      if not isinstance(N, Matrix):
        raise TypeError("Expected input to be of type Matrix")

      if self.shape() != N.shape():
        raise ValueError("Matrices must have the same dimensions to be multiplied")

      new_matrix = [[self.__mat[i][j] * N.__mat[i][j] for j in range(self.__m)] for i in range(self.__n)]

      return Matrix(self.__n, self.__m)._set_matrix(new_matrix)

    def equals(self,N):
      if not isinstance(N, Matrix):
        raise TypeError("Expected input to be of type Matrix")

      if self.shape() != N.shape():
        raise ValueError("Matrices must be of same dimensions")

      for i in range(self.__n):
        for j in range(self.__m):
          if self.__mat[i][j] != N.__mat[i][j]:
            return False
      return True

    def mat_mult(self, N):
      if not isinstance(N, Matrix):
        raise TypeError("Expected input to be of type Matrix")
        
      if self.shape()[1] != N.shape()[0]:
        raise ValueError("Number of columns in first matrix must be equal to number of rows in second matrix")

      result_rows = self.shape()[0]
      result_cols = N.shape()[1]

      result = [[0] * result_cols for _ in range(result_rows)]

      for i in range(result_rows):
        for j in range(result_cols):
          for k in range(self.shape()[1]):
            result[i][j] += self.__mat[i][k] * N.__mat[k][j]
            
      return Matrix(result_rows, result_cols)._set_matrix(result)


In [18]:
##The other functions

def constant(n,m,c):
  mat = [[float(c) for _ in range(m)] for _ in range(n)]
  return mat

def zeros(n,m):
  mat = [[ 0 for _ in range(m)] for _ in range(n)]
  return mat

def ones(n,m):
  mat = [[ 1 for _ in range(m)] for _ in range(n)]
  return mat

def eye(n):
  mat = [[ 0 for _ in range(n)] for _ in range(n)]

  for i in range(n):
    mat[i][i] = 1

  return mat

In [19]:
I = eye(2)
I

[[1, 0], [0, 1]]

In [20]:
A = Matrix(2,2,seed=1)
B = Matrix(2,2,seed=2)
C = Matrix(2,2,seed=3)
print(A)
print("---------")
print(B)
print("---------")
print(C)
print("---------")

  2   9
  1   4
---------
  0   1
  1   5
---------
  3   9
  8   2
---------


In [23]:
##(AB)C = A(BC)
D = C.element_mult(A.element_mult(B))
D

  0  81
  8  40

In [24]:
E = A.element_mult(B.element_mult(C))
E

  0  81
  8  40

In [26]:
##A(B+C) = AB + AC
F = A.element_mult(B.add(C))
F

  6  90
  9  28

In [28]:
G = A.element_mult(B).add(A.element_mult(C))
G

  6  90
  9  28

In [29]:
##AB \neq BA
H = A.mat_mult(B)
H

  9  47
  4  21

In [30]:
J = B.mat_mult(A)
J

  1   4
  7  29

In [31]:
K = A.mat_mult(I)

TypeError: Expected input to be of type Matrix

In [1]:
### Quiz 
def make_deck():
    suites = ["Hearts", "Clubs", "Spades", "Diamonds"]
    cards = [2, 3, 4, 5, 6, 7, 8, 9, 10, "Jack", "Queen", "King", "Ace"]
    
    deck = []  

    for suit in suites:
        for card in cards:
            deck.append((suit, card))  

    return deck


print(make_deck())  


[('Hearts', 2), ('Hearts', 3), ('Hearts', 4), ('Hearts', 5), ('Hearts', 6), ('Hearts', 7), ('Hearts', 8), ('Hearts', 9), ('Hearts', 10), ('Hearts', 'Jack'), ('Hearts', 'Queen'), ('Hearts', 'King'), ('Hearts', 'Ace'), ('Clubs', 2), ('Clubs', 3), ('Clubs', 4), ('Clubs', 5), ('Clubs', 6), ('Clubs', 7), ('Clubs', 8), ('Clubs', 9), ('Clubs', 10), ('Clubs', 'Jack'), ('Clubs', 'Queen'), ('Clubs', 'King'), ('Clubs', 'Ace'), ('Spades', 2), ('Spades', 3), ('Spades', 4), ('Spades', 5), ('Spades', 6), ('Spades', 7), ('Spades', 8), ('Spades', 9), ('Spades', 10), ('Spades', 'Jack'), ('Spades', 'Queen'), ('Spades', 'King'), ('Spades', 'Ace'), ('Diamonds', 2), ('Diamonds', 3), ('Diamonds', 4), ('Diamonds', 5), ('Diamonds', 6), ('Diamonds', 7), ('Diamonds', 8), ('Diamonds', 9), ('Diamonds', 10), ('Diamonds', 'Jack'), ('Diamonds', 'Queen'), ('Diamonds', 'King'), ('Diamonds', 'Ace')]
